# 02 · Os agentes

Demonstra os sistemas respondendo perguntas sobre os anúncios do Airbnb em São Paulo (Inside Airbnb), com o trace de cada consulta e os dados de execução.

| Parte | O que faz | Tempo aproximado |
|---|---|---|
| 1 | Prepara a sessão: recorte, bancos, servidor MCP, Ollama e `qwen3:8b` | ~10 min |
| 2 | Abre a sessão com o servidor MCP e aquece o modelo | ~2 min |
| 3 | **Sistema 1 · agente único** em algumas perguntas da avaliação, com o gabarito ao lado | ~1 min |

**Antes de rodar:** escolha o runtime **T4 GPU** (Ambiente de execução → Alterar o tipo de ambiente de execução). Na Parte 1, ajuste `DRIVE_DIR` para a pasta do Drive com os 8 CSV do recorte.

In [ ]:
# @title Parte 1 · Preparar a sessão
import os, subprocess, sys

REPO_URL = "https://github.com/felipeafigueiredo/text-to-sql-and-cypher.git"
REPO_DIR = "/content/text-to-sql-and-cypher"
DRIVE_DIR = "/content/drive/MyDrive/airbnb_recorte"   # pasta do Drive com os 8 CSV do recorte

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=commit %h · %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, REPO_DIR)

from ambiente.preparar import preparar
TEMPOS = preparar(DRIVE_DIR)

In [ ]:
# @title Parte 2 · Sessão com o servidor MCP e aquecimento do modelo
from agentes.comum import Servidor, aquecer, mostrar
from avaliacao.gabarito import calcular, checagem

if "SRV" in globals():
    await SRV.fechar()                          # se a célula rodar de novo, fecha a sessão anterior
SRV = await Servidor().abrir()                 # sobe o server.py por stdio, lista as ferramentas e lê os esquemas
for t in SRV.ferramentas:
    print(f"🔧 {t.name:<18} {t.description.splitlines()[0]}")
print(f"📄 resources lidos: {list(SRV.esquemas)}")

GABARITO = {p["id"]: p for p in calcular()}    # o gabarito sai dos bancos, pelas consultas de referência
print(f"📋 {len(GABARITO)} perguntas com gabarito")

print(f"\n🔥 aquecimento do modelo (fora das métricas): {await aquecer()} s")

---
## Parte 3 · Sistema 1 · agente único

Um modelo, os dois esquemas no prompt de sistema e as 5 ferramentas, no loop `modelo ⇄ ferramentas`. Se uma consulta dá erro, a mensagem do servidor (com a dica) volta ao modelo, que corrige. É a linha de base: os sistemas 2 e 3 são construídos sobre ele.

```
START ─► modelo ──── pediu ferramenta ───► ferramentas
           ▲                                   │
           └──────────── resultado ────────────┘
           │
           └─► respondeu com texto ─► END
```

Abaixo, sete das 20 perguntas: uma de cada tipo. Cada trace mostra as consultas, o resultado resumido, a resposta, os dados de execução e a **checagem automática** contra o gabarito (números e nomes que a resposta deve ter, e a ferramenta esperada). O veredito final da avaliação é do juiz, no notebook 03.

In [ ]:
# @title Rodar o agente único
from agentes.agente_unico import AgenteUnico

UNICO = AgenteUnico(SRV)
DEMO = ["P01", "P05", "P11", "P15", "P18", "P19", "P20"]

for pid in DEMO:
    p = GABARITO[pid]
    print("═" * 100)
    print(f"{pid} · {p['categoria']} · banco {p['banco']}")
    r = await UNICO.answer(p["pergunta"])
    mostrar(r)
    falhas = checagem(p, r)
    print(f"📋 gabarito: {p['texto']}")
    print(f"{'✅ checagem automática passou' if not falhas else '❌ checagem automática: ' + '; '.join(falhas)}\n")

In [ ]:
# @title Sua pergunta
PERGUNTA = "Quais são os 3 anúncios de Moema com mais avaliações?"  # @param {type:"string"}
mostrar(await UNICO.answer(PERGUNTA))

In [ ]:
# @title Ambiente
import platform
from importlib.metadata import version
import requests
print("Python", platform.python_version())
for lib in ["langgraph", "langchain-core", "langchain-ollama", "langchain-mcp-adapters", "mcp"]:
    print(f"{lib:<24} {version(lib)}")
print("Ollama", requests.get("http://localhost:11434/api/version").json()["version"])
info = requests.post("http://localhost:11434/api/show", json={"model": os.environ["AIRBNB_MODELO"]}).json()["details"]
print(f"modelo {os.environ['AIRBNB_MODELO']} · {info['parameter_size']} · {info['quantization_level']}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -h | head -n 2",
                     shell=True, capture_output=True, text=True).stdout)
print("⏱ preparação (s):", TEMPOS)